# Lab: Modern multi-period DiD (Python)

[View this lab on the QED Labs website](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-modern-estimators-python-lab.html)

## How to use this lab

Allow 45–60 minutes. Basic regression and Python data-frame familiarity are assumed.
Use the [tested environment setup](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-reproducibility.html) before running every cell in order.

[R companion](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-modern-estimators-lab.html) · [Application catalogue](https://defenceeconomist.github.io/qedlabs/notes/did/difference-in-differences-applications.html)

The code downloads a checksum-verified upstream data file on first use and caches it locally. This is a teaching reproduction, not a replication of every specification in the original paper.

## Research question and estimand

Use the `did` package's county panel of teen employment and minimum-wage adoption. This is a compact teaching dataset, not the complete underlying policy replication. Estimate $ATT(g,t)$ for counties first treated in year $g$, then state how those effects are averaged [@callaway2021multipletime].

## 1. Load the panel

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr

# Immutable upstream package data; do not silently accept a changed file.
url = "https://raw.githubusercontent.com/cran/did/453a1be859411edc1c8933891940cb3a570d7095/data/mpdta.rda"
expected_sha256 = "0be3bc5eaaedaa50e047deaf45ad0e193e7210df1644103c63a6c1baa75965d4"
cache = Path(".qedlabs-cache")
cache.mkdir(exist_ok=True)
path = cache / "mpdta.rda"
if not path.exists():
    payload = urlopen(url, timeout=60).read()
    assert hashlib.sha256(payload).hexdigest() == expected_sha256, "Data checksum mismatch"
    path.write_bytes(payload)
assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha256, "Cached data changed"
data = pyreadr.read_r(str(path))["mpdta"]
print({p: metadata.version(p) for p in ["pandas", "numpy", "pyreadr"]})
print(data.shape)

In [ ]:
from differences import ATTgt
from scipy.stats import norm
panel = data.copy().sort_values(['countyreal', 'year'])
assert not panel.duplicated(['countyreal', 'year']).any()
assert panel[['countyreal', 'year', 'first.treat', 'lemp', 'lpop']].notna().all().all()
assert panel.groupby('countyreal').size().nunique() == 1
assert panel.groupby('countyreal')['first.treat'].nunique().eq(1).all()
# In mpdta, treat marks ever-treated counties, not current treatment exposure.
panel['exposed'] = (panel['first.treat'] > 0) & (panel.year >= panel['first.treat'])
assert panel.groupby('countyreal').exposed.diff().dropna().ge(0).all()
print(panel.groupby('first.treat').countyreal.nunique())
print({'rows': len(panel), 'counties': panel.countyreal.nunique(), 'years': panel.year.nunique()})
# differences uses missing cohort values for never-treated units.
python_panel = panel.copy()
python_panel['cohort'] = python_panel['first.treat'].replace(0, np.nan)
python_panel = python_panel.set_index(['countyreal', 'year'])

## 2. Group-time effects and inference

Match R's universal base period, zero anticipation, never-treated controls, and doubly robust estimation. Panel inference clusters on the county entity by default. County clustering demonstrates software behavior; state-level minimum-wage assignment can induce dependence across counties and requires assignment-level identifiers for policy inference.

In [ ]:
att_never = ATTgt(python_panel, cohort_column='cohort', base_period='universal', anticipation=0)
raw = att_never.fit('lemp', est_method='dr', control_group='never_treated',
                   boot_iterations=0, progress_bar=False)

def flat_result(table):
    out = table.to_pandas().copy() if hasattr(table, "to_pandas") else table.copy()
    out.columns = [next(str(part) for part in column if str(part) in ['ATT', 'std_error', 'lower', 'upper', 'zero_not_in_cband'])
                   for column in out.columns]
    return out

gt = flat_result(raw)
print(gt)
assert np.isfinite(gt.ATT).all()

**Pinned-version limitation:** `differences` 0.3.0 returns nonfinite bootstrap bands when universal-base reference cells have zero variance. This lab uses analytic standard errors and explicit conservative Bonferroni simultaneous bands over the displayed, non-reference coefficients. R uses multiplier-bootstrap bands. Their intervals are not expected to match.

## 3. Aggregation and simultaneous bands

In [ ]:
aggregations = {kind: flat_result(att_never.aggregate(kind, boot_iterations=0))
                for kind in ['simple', 'cohort', 'event', 'time']}
for kind, table in aggregations.items():
    print(kind, '\n', table)
dynamic = aggregations['event'].copy()
# Omitted -1 is a normalization, not an estimated coefficient.
dynamic = dynamic.loc[dynamic.index != -1].copy()
assert np.isfinite(dynamic[['ATT', 'std_error']]).all().all()
critical = norm.ppf(1 - 0.05 / (2 * len(dynamic)))
dynamic['simult_lower'] = dynamic.ATT - critical * dynamic.std_error
dynamic['simult_upper'] = dynamic.ATT + critical * dynamic.std_error
ax = plt.subplots(figsize=(9, 5))[1]
ax.errorbar(dynamic.index, dynamic.ATT, yerr=critical * dynamic.std_error, marker='o')
ax.axhline(0, color='grey'); ax.axvline(-1, linestyle='--', color='grey')
ax.set(xlabel='Years relative to treatment', ylabel='Log-employment ATT',
       title='95% Bonferroni bands over displayed event coefficients')
plt.show()
print(dynamic)
assert np.isclose(aggregations['simple'].ATT.iloc[0], -0.039951, atol=1e-6)

The simple average weights supported post-treatment cells by cohort size. Event-time estimates average different cohorts at different horizons. The Bonferroni bands control a family of comparisons asymptotically and are generally conservative; neither band construction removes confounding.

## 4. Alternative controls and conditional trends

In [ ]:
att_notyet = ATTgt(python_panel, cohort_column='cohort', base_period='universal')
att_notyet.fit('lemp', est_method='dr', control_group='not_yet_treated', boot_iterations=0, progress_bar=False)
att_adjusted = ATTgt(python_panel, cohort_column='cohort', base_period='universal')
att_adjusted.fit('lemp ~ lpop', est_method='dr', control_group='never_treated',
                 base_delta='base', boot_iterations=0, progress_bar=False)
comparison = pd.DataFrame({
    'never': flat_result(att_never.aggregate('simple')).ATT,
    'not_yet': flat_result(att_notyet.aggregate('simple')).ATT,
    'conditional_lpop': flat_result(att_adjusted.aggregate('simple')).ATT,
})
print(comparison)

Future adopters require a credible untreated comparison and no anticipation before their adoption. Conditioning on baseline log population invokes conditional parallel trends and overlap; it does not guarantee them. Package nuisance fitting can differ, so unconditional group-time estimates are the primary cross-language equality check.

## R extension: sensitivity analysis

The [R extension](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-modern-estimators-lab.html#optional-extension-sensitivity-to-parallel-trends-violations) restricts to the 2006 cohort and never-treated counties, then uses `HonestDiD`. It studies the first post-treatment effect, not the full staggered aggregate. Its relative-magnitude restriction bounds post-treatment changes in the untreated group gap relative to the largest pre-treatment change. Sampling uncertainty in that calibration is part of the procedure [@rambachanroth2023parallel].

Read the verified extension output alongside the [sensitivity notes](https://defenceeconomist.github.io/qedlabs/notes/did/rambachan-roth-sensitivity-notes.html). State the cohort, target period, restriction, and first tested bound admitting zero. A coarse grid does not establish an exact breakdown threshold.

## Worked answers

1. **Why not one pooled coefficient?** Cohorts have different effects and observed exposure windows. Aggregation is part of the policy question.
2. **Why might the control choice matter?** Not-yet-treated units change support and the counterfactual assumption; they are not automatically better controls.
3. **Why universal baselines?** Pre-treatment estimates then compare to the last untreated period rather than adjacent-period changes. This affects plot interpretation.
4. **Why different intervals across languages?** This Python lab uses analytic variances and Bonferroni critical values; the R lab uses multiplier bootstrap. Equality of point estimates does not imply identical inference.
5. **What does double robustness protect?** Under identification and overlap, consistency can survive misspecification of one nuisance model. It does not protect against failure of parallel trends.